1. Introduction

This project aims to predict whether passengers survived the Titanic disaster using machine learning.
The goal is to explore passenger data, identify factors associated with survival, perform data preprocessing and feature engineering, and build predictive models.
Several machine learning models and feature combinations are evaluated to identify a model with good predictive performance.

2. Data Loading
The training and test datasets are loaded using pandas.
- train.csv: Contains passenger information and the target variable (Survived).
- test.csv: Contains passenger information used to generate survival predictions.
The training dataset is used for exploratory data analysis, model training, and evaluation, while the test dataset is used for the final prediction.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
#データの読み込み
import pandas as pd
train=pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
print(train.head(5))
test=pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


今回のデータは名前、生存結果(Survived)、乗客クラス(Pclass)、性別、年齢、兄弟・配偶者の搭乗人数、親子の搭乗人数(Parch)、チケットの番号、運賃、客室番号(Cabin)、乗車港(Embarked)である。

In [3]:
#scvの名称を確認
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


3. Exploratory Data Analysis (EDA)

Exploratory data analysis (EDA) was conducted to understand the dataset and identify factors associated with passenger survival.
The following aspects were investigated:
Overall survival rate
Survival rate by sex
Survival rate by passenger class (Pclass)
Age distribution and survival rate by age group
Family relationships (SibSp and Parch)
Family size (FamilySize) and survival rate
These analyses were used to understand the characteristics of the passengers and guide feature engineering and model selection.

In [4]:
#データの基本情報の確認
print(train.shape)
print(train.dtypes)
print(train.isnull().sum())

(891, 12)
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [5]:
#生存率の確認
print(train["Survived"].value_counts())
生存率=((train["Survived"]==1).sum()/len(train["Survived"]))*100
print("生存率:",生存率)
#性別と生存確率の関係
print(train[train["Survived"]==1]["Sex"].value_counts())
男性の生存率=len(train.loc[(train["Survived"] == 1)&(train["Sex"] == "male")])/len(train[train["Sex"]=="male"]["Survived"])*100
女性の生存率=len(train.loc[(train["Survived"] == 1)&(train["Sex"] == "female")])/len(train[train["Sex"]=="female"]["Survived"])*100
print("男性の生存率:",男性の生存率)
print("女性の生存率:",女性の生存率)
#Pclassと生存率の関係
print(train[train["Survived"]==1]["Pclass"].value_counts())
Pclass1の生存率=len(train.loc[(train["Survived"] == 1)&(train["Pclass"] == 1)])/len(train[train["Pclass"]==1]["Survived"])*100
Pclass2の生存率=len(train.loc[(train["Survived"] == 1)&(train["Pclass"] == 2)])/len(train[train["Pclass"]==2]["Survived"])*100
Pclass3の生存率=len(train.loc[(train["Survived"] == 1)&(train["Pclass"] == 3)])/len(train[train["Pclass"]==3]["Survived"])*100
print("Pclass1の生存率:",Pclass1の生存率)
print("Pclass2の生存率:",Pclass2の生存率)
print("Pclass3の生存率:",Pclass3の生存率)

Survived
0    549
1    342
Name: count, dtype: int64
生存率: 38.38383838383838
Sex
female    233
male      109
Name: count, dtype: int64
男性の生存率: 18.890814558058924
女性の生存率: 74.20382165605095
Pclass
1    136
3    119
2     87
Name: count, dtype: int64
Pclass1の生存率: 62.96296296296296
Pclass2の生存率: 47.28260869565217
Pclass3の生存率: 24.236252545824847


生存率は38%だった。生存した数が低かった。男性よりも女性の方が生存した数が多く、生存した女性の数は男性のほぼ２倍だった。また、それぞれの性別における生存割合からも女性が優先的に避難することができたと推定することができる。更に、乗客のクラスがいい方が生存した数は多かった。また、それぞれの乗客のクラスの生存率を比較すると乗客クラスによって避難のしやすや変化したり、または乗客のクラスが上の人の方が優先的に避難することができたと推測することができる。

In [6]:
#EDA
#年齢と生存確率の関係
print(train["Age"].describe())
print(train["Age"].isnull().sum())

bins=[0,12,18,30,50,100]
labels=["子供","10代","20代","中年層","51歳以上"]
Age_distribution=pd.cut(train["Age"],bins=bins,labels=labels,right=False)
print(Age_distribution.value_counts())

子供の生存確率=len(train[(train["Survived"] == 1)&(Age_distribution == "子供")])/len(train[Age_distribution == "子供"]["Survived"])*100
十代の生存確率=len(train[(train["Survived"] == 1)&(Age_distribution == "10代")])/len(train[Age_distribution == "10代"]["Survived"])*100
二十代の生存確率=len(train[(train["Survived"] == 1)&(Age_distribution == "20代")])/len(train[Age_distribution == "20代"]["Survived"])*100
中年層の生存確率=len(train[(train["Survived"] == 1)&(Age_distribution == "中年層")])/len(train[Age_distribution == "中年層"]["Survived"])*100
五十一歳以上の生存確率=len(train[(train["Survived"] == 1)&(Age_distribution == "51歳以上")])/len(train[Age_distribution == "51歳以上"]["Survived"])*100
print("子供の生存確率:",子供の生存確率)
print("十代の生存確率:",十代の生存確率)
print("二十代の生存確率:",二十代の生存確率)
print("中年層の生存確率:",中年層の生存確率)
print("五十一歳以上の生存確率:",五十一歳以上の生存確率)







count    714.000000
mean      29.699118
std       14.526497
min        0.420000
25%       20.125000
50%       28.000000
75%       38.000000
max       80.000000
Name: Age, dtype: float64
177
Age
20代      271
中年層      256
51歳以上     74
子供        68
10代       45
Name: count, dtype: int64
子供の生存確率: 57.35294117647059
十代の生存確率: 48.888888888888886
二十代の生存確率: 35.05535055350554
中年層の生存確率: 41.796875
五十一歳以上の生存確率: 36.486486486486484


生存した数を比較すると20代が最も多く10代が最も多いがそれぞれの年齢の生存割合を調べてみると年齢が若い方が生存割合が高くなっていた。そのため、小学生や小学生以下の子供が優先的に避難することができたのではないかと予測できる。

In [7]:
#家族構成と生存率の関係
print(train["SibSp"].value_counts())
print(train["Parch"].value_counts())

result_SibSp=train.groupby("SibSp")["Survived"].apply(lambda group: (group.sum()/len(group))*100)
result_Parch=train.groupby("Parch")["Survived"].apply(lambda group: (group.sum()/len(group))*100)
print("兄弟の生存確率:",result_SibSp)
print("家族の生存確率:",result_Parch)

SibSp
0    608
1    209
2     28
4     18
3     16
8      7
5      5
Name: count, dtype: int64
Parch
0    678
1    118
2     80
5      5
3      5
4      4
6      1
Name: count, dtype: int64
兄弟の生存確率: SibSp
0    34.539474
1    53.588517
2    46.428571
3    25.000000
4    16.666667
5     0.000000
8     0.000000
Name: Survived, dtype: float64
家族の生存確率: Parch
0    34.365782
1    55.084746
2    50.000000
3    60.000000
4     0.000000
5    20.000000
6     0.000000
Name: Survived, dtype: float64


兄弟・および親子の同乗人数別に生存率を比較したところ、どちらも同乗者が0人の場合より、1人の場合の生存率が高かった。一方、同乗人数が多いグループでは生存率が低くなる傾向もあった。

In [8]:
#家族の人数と生存率
train["FamilySize"]=train["SibSp"]+train["Parch"]+1
print(train["FamilySize"].value_counts())

result_FamilySize=train.groupby("FamilySize")["Survived"].apply(lambda group: (group.sum()/len(group))*100)
print("家族の人数と生存確率:",result_FamilySize)

FamilySize
1     537
2     161
3     102
4      29
6      22
5      15
7      12
11      7
8       6
Name: count, dtype: int64
家族の人数と生存確率: FamilySize
1     30.353818
2     55.279503
3     57.843137
4     72.413793
5     20.000000
6     13.636364
7     33.333333
8      0.000000
11     0.000000
Name: Survived, dtype: float64


一人できた乗客が圧倒的に多く、二人から三人できた乗客も比較的多い。家族が多い乗客ほど人数が少なかった。家族が多い人が生存率が高いや一人だと生存確率といった家族の人数による生存確率の傾向は見られなかった。
性別と客室クラスは生存確率の差が大きく予測するために重要な特徴量だと予測する。また、年齢や家族構成も予測に影響すると考える。

In [9]:
#Ageの欠損値の処理
train.isnull().sum()
print(train.loc[train["Age"].isna()]["Pclass"].value_counts())

print("Pclass1:",train.loc[train["Pclass"]==1]["Age"].mean())
print("Pclass2:",train.loc[train["Pclass"]==2]["Age"].mean())
print("Pclass3:",train.loc[train["Pclass"]==3]["Age"].mean())

print("男性:",train.loc[train["Sex"]=="male"]["Age"].mean())
print("女性:",train.loc[train["Sex"]=="female"]["Age"].mean())

print("Pclass1_男性:",train.loc[(train["Pclass"]==1) & (train["Sex"]=="male")]["Age"].mean())
print("Pclass2_男性:",train.loc[(train["Pclass"]==2) & (train["Sex"]=="male")]["Age"].mean())
print("Pclass3_男性:",train.loc[(train["Pclass"]==3) & (train["Sex"]=="male")]["Age"].mean())

print("Pclass1_女性:",train.loc[(train["Pclass"]==1) & (train["Sex"]=="female")]["Age"].mean())
print("Pclass2_女性:",train.loc[(train["Pclass"]==2) & (train["Sex"]=="female")]["Age"].mean())
print("Pclass3_女性:",train.loc[(train["Pclass"]==3) & (train["Sex"]=="female")]["Age"].mean())

Pclass1_男性_中央値=train.loc[(train["Pclass"]==1) & (train["Sex"]=="male")]["Age"].median()
Pclass2_男性_中央値=train.loc[(train["Pclass"]==2) & (train["Sex"]=="male")]["Age"].median()
Pclass3_男性_中央値=train.loc[(train["Pclass"]==3) & (train["Sex"]=="male")]["Age"].median()

Pclass1_女性_中央値=train.loc[(train["Pclass"]==1) & (train["Sex"]=="female")]["Age"].median()
Pclass2_女性_中央値=train.loc[(train["Pclass"]==2) & (train["Sex"]=="female")]["Age"].median()
Pclass3_女性_中央値=train.loc[(train["Pclass"]==3) & (train["Sex"]=="female")]["Age"].median()

print("Pclass1_男性_中央値:",train.loc[(train["Pclass"]==1) & (train["Sex"]=="male")]["Age"].median())
print("Pclass2_男性_中央値:",train.loc[(train["Pclass"]==2) & (train["Sex"]=="male")]["Age"].median())
print("Pclass3_男性_中央値:",train.loc[(train["Pclass"]==3) & (train["Sex"]=="male")]["Age"].median())

print("Pclass1_女性_中央値:",train.loc[(train["Pclass"]==1) & (train["Sex"]=="female")]["Age"].median())
print("Pclass2_女性_中央値:",train.loc[(train["Pclass"]==2) & (train["Sex"]=="female")]["Age"].median())
print("Pclass3_女性_中央値:",train.loc[(train["Pclass"]==3) & (train["Sex"]=="female")]["Age"].median())

train.loc[(train["Pclass"]==1) & (train["Sex"]=="male"),"Age"]=train.loc[(train["Pclass"]==1) & (train["Sex"]=="male"),"Age"].fillna(Pclass1_男性_中央値)
train.loc[(train["Pclass"]==2) & (train["Sex"]=="male"),"Age"]=train.loc[(train["Pclass"]==2) & (train["Sex"]=="male"),"Age"].fillna(Pclass2_男性_中央値)
train.loc[(train["Pclass"]==3) & (train["Sex"]=="male"),"Age"]=train.loc[(train["Pclass"]==3) & (train["Sex"]=="male"),"Age"].fillna(Pclass3_男性_中央値)

train.loc[(train["Pclass"]==1) & (train["Sex"]=="female"),"Age"]=train.loc[(train["Pclass"]==1) & (train["Sex"]=="female"),"Age"].fillna(Pclass1_女性_中央値)
train.loc[(train["Pclass"]==2) & (train["Sex"]=="female"),"Age"]=train.loc[(train["Pclass"]==2) & (train["Sex"]=="female"),"Age"].fillna(Pclass2_女性_中央値)
train.loc[(train["Pclass"]==3) & (train["Sex"]=="female"),"Age"]=train.loc[(train["Pclass"]==3) & (train["Sex"]=="female"),"Age"].fillna(Pclass3_女性_中央値)




Pclass
3    136
1     30
2     11
Name: count, dtype: int64
Pclass1: 38.233440860215055
Pclass2: 29.87763005780347
Pclass3: 25.14061971830986
男性: 30.72664459161148
女性: 27.915708812260537
Pclass1_男性: 41.28138613861386
Pclass2_男性: 30.74070707070707
Pclass3_男性: 26.507588932806325
Pclass1_女性: 34.61176470588235
Pclass2_女性: 28.722972972972972
Pclass3_女性: 21.75
Pclass1_男性_中央値: 40.0
Pclass2_男性_中央値: 30.0
Pclass3_男性_中央値: 25.0
Pclass1_女性_中央値: 35.0
Pclass2_女性_中央値: 28.0
Pclass3_女性_中央値: 21.5


予測精度において性別と乗客クラスが大きく影響していると考察したため、それぞれのクラスと性別の組み合わせにおける平均値と中央値を計算して中央値で欠損値を補った。

In [10]:
#Embarkedの欠損値の処理
train["Embarked"].value_counts()
train["Embarked"]=train["Embarked"].fillna("S")
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
FamilySize       0
dtype: int64

Embarkedは2件しか欠損値がなかったので、最頻値Sで欠損値を補った。

In [11]:
#Cabinの欠損値の処理
train["New_Cabin"]=train.loc[train["Cabin"].notna(),"Cabin"]
train["Deck"]=train["New_Cabin"].str[0]
train["Deck"].value_counts()
print(train[["Deck","Pclass"]].value_counts())

print("Aの生存率:",train.loc[train["Deck"]=="A"]["Survived"].mean())
print("Bの生存率:",train.loc[train["Deck"]=="B"]["Survived"].mean())
print("Cの生存率:",train.loc[train["Deck"]=="C"]["Survived"].mean())
print("Dの生存率:",train.loc[train["Deck"]=="D"]["Survived"].mean())
print("Eの生存率:",train.loc[train["Deck"]=="E"]["Survived"].mean())
print("Fの生存率:",train.loc[train["Deck"]=="F"]["Survived"].mean())
print("Gの生存率:",train.loc[train["Deck"]=="G"]["Survived"].mean())
print("Tの生存率:",train.loc[train["Deck"]=="T"]["Survived"].mean())

train["Deck"]=train["Deck"].fillna("Unknown")
train.isnull().sum()

Deck  Pclass
C     1         59
B     1         47
D     1         29
E     1         25
A     1         15
F     2          8
      3          5
D     2          4
G     3          4
E     2          4
      3          3
T     1          1
Name: count, dtype: int64
Aの生存率: 0.4666666666666667
Bの生存率: 0.7446808510638298
Cの生存率: 0.5932203389830508
Dの生存率: 0.7575757575757576
Eの生存率: 0.75
Fの生存率: 0.6153846153846154
Gの生存率: 0.5
Tの生存率: 0.0


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
FamilySize       0
New_Cabin      687
Deck             0
dtype: int64

客室番号の先頭の文字が客室のデッキを表していたので、客室のデッキとPclassの関係性がないか調べた。すると、AからC、Tは1等でDからFは2等と3等だった。デッキと生存確率を調べるとデッキによって等級が異なるなどの要因から生存確率が変化した。そのため、特徴量Deckは予測の精度に影響を与えると考察した。

5. Model Training & Evaluation
Several machine learning models were trained and evaluated using different feature combinations. Model performance was compared using accuracy to identify the most suitable mode

In [12]:
#特徴量Sex、Pclass、Age時の学習
train["Sex"]=train["Sex"].map({"female":1 ,"male":0})
x=train[["Sex","Pclass","Age"]]
y=train["Survived"]

from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,random_state=42
)
from sklearn.linear_model import LogisticRegression 
model=LogisticRegression()
model.fit(x_train,y_train)
y_pred=model.predict(x_test)

from sklearn.metrics import  accuracy_score
accuracy=accuracy_score(y_test,y_pred)

print("正確性:",accuracy)

train_score=model.score(x_train,y_train)
test_score=model.score(x_test,y_test)
print("訓練データの正解率:",train_score)
print("テストデータの正解率:",test_score)

正確性: 0.8100558659217877
訓練データの正解率: 0.7949438202247191
テストデータの正解率: 0.8100558659217877


性別、等級、年齢の三つの特徴量で正確性と正解率を計算したがそれぞれ80%程度で、過学習も起きていなかった。特徴量を増やしていってモデルを改善する。

In [13]:
#FamilySizeを追加
x=train[["Sex","Pclass","Age","FamilySize"]]
y=train["Survived"]


from sklearn.model_selection import train_test_split
x_train1,x_test1,y_train1,y_test1=train_test_split(
    x,y,test_size=0.2,random_state=42
)
from sklearn.linear_model import LogisticRegression 
model1=LogisticRegression()
model1.fit(x_train1,y_train1)
y_pred1=model1.predict(x_test1)

from sklearn.metrics import  accuracy_score
accuracy=accuracy_score(y_test1,y_pred1)

print("正確性:",accuracy)

train_score1=model1.score(x_train1,y_train1)
test_score1=model1.score(x_test1,y_test1)
print("訓練データの正解率:",train_score1)
print("テストデータの正解率:",test_score1)

正確性: 0.8324022346368715
訓練データの正解率: 0.8033707865168539
テストデータの正解率: 0.8324022346368715


FamilySizeを追加することで正確性と正解率も向上した。

In [14]:
#Fareを追加
x=train[["Sex","Pclass","Age","FamilySize","Fare"]]
y=train["Survived"]


from sklearn.model_selection import train_test_split
x_train2,x_test2,y_train2,y_test2=train_test_split(
    x,y,test_size=0.2,random_state=42
)
from sklearn.linear_model import LogisticRegression 
model2=LogisticRegression()
model2.fit(x_train2,y_train2)
y_pred2=model2.predict(x_test2)

from sklearn.metrics import  accuracy_score
accuracy2=accuracy_score(y_test2,y_pred2)

print("正確性:",accuracy2)

train_score2=model2.score(x_train2,y_train2)
test_score2=model2.score(x_test2,y_test2)
print("訓練データの正解率:",train_score2)
print("テストデータの正解率:",test_score2)

正確性: 0.8268156424581006
訓練データの正解率: 0.8033707865168539
テストデータの正解率: 0.8268156424581006


6. Feature Selection & Experimentation
Different feature combinations were tested, and feature selection techniques were explored to evaluate their impact on model performance.

In [15]:
#One-Hot Encoding
a=pd.get_dummies(train["Embarked"])
b=pd.get_dummies(train["Deck"])
print(a.shape)
print(a.head(5))

print(b.shape)
print(b.head(5))

a = pd.get_dummies(train["Embarked"], prefix="Embarked")
b = pd.get_dummies(train["Deck"], prefix="Deck")
c = pd.concat([a, b], axis=1)
train_new = pd.concat([train, c], axis=1)


(891, 3)
       C      Q      S
0  False  False   True
1   True  False  False
2  False  False   True
3  False  False   True
4  False  False   True
(891, 9)
       A      B      C      D      E      F      G      T  Unknown
0  False  False  False  False  False  False  False  False     True
1  False  False   True  False  False  False  False  False    False
2  False  False  False  False  False  False  False  False     True
3  False  False   True  False  False  False  False  False    False
4  False  False  False  False  False  False  False  False     True


In [16]:
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

model = LogisticRegression(max_iter=100000)

candidates = [
    "Sex", "Pclass", "Age", "FamilySize", "Fare",
    "Embarked_C", "Embarked_Q", "Embarked_S",
    "Deck_A", "Deck_B", "Deck_C", "Deck_D", "Deck_E", "Deck_F", "Deck_G", "Deck_T", "Deck_Unknown"
]

x_all = train_new[candidates]
y_all = train_new["Survived"]


selector = RFECV(
    estimator=model,
    step=1,                                     
    cv=StratifiedKFold(5, shuffle=True, random_state=42), 
    scoring="accuracy"                          
)

selector.fit(x_all, y_all)

best_pattern = list(x_all.columns[selector.support_])
best_score = selector.cv_results_["mean_test_score"].max()

print("選ばれた列の数:", selector.n_features_)
print("最も良い組み合わせ:", best_pattern)
print("最も良い正確率:",best_score)

選ばれた列の数: 16
最も良い組み合わせ: ['Sex', 'Pclass', 'Age', 'FamilySize', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'Deck_Unknown']
最も良い正確率: 0.8058251208335949


どの特徴量の組み合わせが最も良いかをRFECVで調べた結果16の特徴量の組み合わせになった。

In [17]:
#最終確認

x=train_new[["Sex","Pclass","Age","FamilySize","Embarked_C","Embarked_Q","Embarked_S","Deck_A","Deck_B","Deck_C","Deck_D","Deck_E","Deck_F","Deck_G","Deck_T","Deck_Unknown"]]
y=train_new["Survived"]


from sklearn.model_selection import train_test_split
x_train3,x_test3,y_train3,y_test3=train_test_split(
    x,y,test_size=0.2,random_state=42
)
from sklearn.linear_model import LogisticRegression 
model3=LogisticRegression()
model3.fit(x_train3,y_train3)
y_pred3=model3.predict(x_test3)

from sklearn.metrics import  accuracy_score
accuracy=accuracy_score(y_test3,y_pred3)

print("正確性:",accuracy)

train_score3=model3.score(x_train3,y_train3)
test_score3=model3.score(x_test3,y_test3)
print("訓練データの正解率:",train_score3)
print("テストデータの正解率:",test_score3)

正確性: 0.8212290502793296
訓練データの正解率: 0.8089887640449438
テストデータの正解率: 0.8212290502793296


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


RFECVでの組み合わせで正解率と正確性を調べたが最も高いモデルではなかった。今回のRFECVでは標準化していなかったため、データによって範囲が大きく異なってしまい、データの与える影響が大きく変化してしまったため、今回の結果になってしまったと考察する。最も精度が良かった組み合わせは["Sex", "Pclass", "Age","FamilySize"]でこの組み合わせのモデルでテストデータそして提出用データを予測する。

7. Final Model
The final model was selected based on the evaluation results of the previous experiments. The selected features were used to train the model and generate predictions for the test dataset.

In [18]:
#テストデータの処理
test["FamilySize"]=test["SibSp"]+test["Parch"]+1
print(test[["Sex", "Pclass", "Age","FamilySize"]].head())
print(test[["Sex", "Pclass", "Age"]].isnull().sum())
print(test.shape)

test.loc[(test["Pclass"]==1) & (test["Sex"]=="male"),"Age"]=test.loc[(test["Pclass"]==1) & (test["Sex"]=="male"),"Age"].fillna(Pclass1_男性_中央値)
test.loc[(test["Pclass"]==2) & (test["Sex"]=="male"),"Age"]=test.loc[(test["Pclass"]==2) & (test["Sex"]=="male"),"Age"].fillna(Pclass2_男性_中央値)
test.loc[(test["Pclass"]==3) & (test["Sex"]=="male"),"Age"]=test.loc[(test["Pclass"]==3) & (test["Sex"]=="male"),"Age"].fillna(Pclass3_男性_中央値)

test.loc[(test["Pclass"]==1) & (test["Sex"]=="female"),"Age"]=test.loc[(test["Pclass"]==1) & (test["Sex"]=="female"),"Age"].fillna(Pclass1_女性_中央値)
test.loc[(test["Pclass"]==2) & (test["Sex"]=="female"),"Age"]=test.loc[(test["Pclass"]==2) & (test["Sex"]=="female"),"Age"].fillna(Pclass2_女性_中央値)
test.loc[(test["Pclass"]==3) & (test["Sex"]=="female"),"Age"]=test.loc[(test["Pclass"]==3) & (test["Sex"]=="female"),"Age"].fillna(Pclass3_女性_中央値)

test["Sex"]=test["Sex"].map({"female":1 ,"male":0})

y_pred_last=model1.predict(x_test1)
print(y_pred_last[:10])
print(x_train1.columns)

      Sex  Pclass   Age  FamilySize
0    male       3  34.5           1
1  female       3  47.0           2
2    male       2  62.0           1
3    male       3  27.0           1
4  female       3  22.0           3
Sex        0
Pclass     0
Age       86
dtype: int64
(418, 12)
[0 0 0 1 1 1 1 0 1 1]
Index(['Sex', 'Pclass', 'Age', 'FamilySize'], dtype='object')


8. Submission
Predictions were generated for the test dataset and saved as a CSV file in the required format for submission to Kaggle.

In [19]:
#提出用 
x_submit = test[["Sex", "Pclass", "Age","FamilySize"]]
y_pred_sub=model1.predict(x_submit)
print(y_pred_sub.shape)
print(y_pred_sub[:10])
submission=pd.DataFrame({
    'PassengerId':test["PassengerId"],
    'Survived':y_pred_sub
})
print(submission.shape)
print(submission[:10])
submission.to_csv("submission.csv", index=False)
from IPython.display import FileLink
FileLink("submission.csv")

(418,)
[0 0 0 0 1 0 1 0 1 0]
(418, 2)
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
5          897         0
6          898         1
7          899         0
8          900         1
9          901         0


/kaggle/working/submission.csv